# Sentiment Analysis of Customer Reviews using VADER and TextBlob

This notebook performs a comparative sentiment analysis using two popular Python libraries: **VADER** (from NLTK) and **TextBlob**.  
It processes customer reviews, applies both sentiment analyzers, classifies the results, and prepares the data for evaluation.

In [1]:
import pandas as pd

# Path to the CSV file
data_file_path = "../1_data_collection/reviews_extracting/trustpilot_reviews_final.csv"

# Load the data into a DataFrame
df = pd.read_csv(data_file_path)

# Preview
df.head()

,name,country,rating,title,text,date_of_experience,has_reply,company
0,Przemysław Rosuł,PL,1,Ignoring the specified pickup time,Ignoring the specified pickup time. Providing ...,2025-05-13,0,www.viator.com
1,kathy mrozek,US,5,Athens Evening food tour with Katrina,"Katrina, the tour guide was fabulous! She mad...",2025-05-15,0,www.viator.com
2,MARY MURRAY,US,5,All you need for travel with confidence,A great app! My excursions have been awesome! ...,2025-05-12,0,www.viator.com
3,K Shay Smith,MX,1,I booked the excursion and paid the…,I booked the excursion and paid the money up f...,2025-05-12,0,www.viator.com
4,Jeff Paine,US,5,Great food tour of Prague. Great conversation ...,Great food tour of Prague,2025-05-07,0,www.viator.com


In [7]:
import nltk
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/ubuntu/nltk_data...


True

## 2. Sentiment Analysis using VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon and rule-based sentiment analysis tool 
that is particularly sensitive to sentiment expressed in social media and short texts.


In [8]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Initialize VADER analyzer
analyzer = SentimentIntensityAnalyzer()

# Apply VADER to each review
def vader_sentiment(text):
    if pd.isna(text):
        return {'neg': None, 'neu': None, 'pos': None, 'compound': None}
    return analyzer.polarity_scores(text)

df['vader_sentiment'] = df['text'].apply(vader_sentiment)

# Extract individual scores
df['vader_compound'] = df['vader_sentiment'].apply(lambda x: x['compound'] if x else None)
df['vader_neg'] = df['vader_sentiment'].apply(lambda x: x['neg'] if x else None)
df['vader_neu'] = df['vader_sentiment'].apply(lambda x: x['neu'] if x else None)
df['vader_pos'] = df['vader_sentiment'].apply(lambda x: x['pos'] if x else None)


### Classifying VADER Sentiment

We classify each review into three categories:
- **Positive**: Compound score ≥ 0.05  
- **Negative**: Compound score ≤ -0.05  
- **Neutral**: Everything in between


In [9]:
def classify_vader_sentiment(score):
    if score is None:
        return None
    elif score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['vader_sentiment_category'] = df['vader_compound'].apply(classify_vader_sentiment)

# Display example
df[['text', 'vader_compound', 'vader_sentiment_category']].head()

,text,vader_compound,vader_sentiment_category
0,Ignoring the specified pickup time. Providing ...,-0.7013,negative
1,"Katrina, the tour guide was fabulous! She mad...",0.8682,positive
2,A great app! My excursions have been awesome! ...,0.9440,positive
3,I booked the excursion and paid the money up f...,-0.5574,negative
4,Great food tour of Prague,0.6249,positive


## 3. Sentiment Analysis using TextBlob

TextBlob is a simple NLP library that provides a straightforward API for diving into common natural language processing tasks, including sentiment analysis. It returns:
- **Polarity**: ranges from -1 (negative) to +1 (positive)
- **Subjectivity**: ranges from 0 (objective) to 1 (subjective)

In [10]:
from textblob import TextBlob

# Apply TextBlob to each review
def textblob_sentiment(text):
    if pd.isna(text):
        return {'polarity': None, 'subjectivity': None}
    blob = TextBlob(text)
    return {'polarity': blob.sentiment.polarity, 'subjectivity': blob.sentiment.subjectivity}

df['textblob_sentiment'] = df['text'].apply(textblob_sentiment)

# Extract scores
df['textblob_polarity'] = df['textblob_sentiment'].apply(lambda x: x['polarity'] if x else None)
df['textblob_subjectivity'] = df['textblob_sentiment'].apply(lambda x: x['subjectivity'] if x else None)


In [11]:
def classify_textblob_sentiment(score):
    if score is None:
        return None
    elif score > 0:
        return 'positive'
    elif score < 0:
        return 'negative'
    else:
        return 'neutral'

df['textblob_sentiment_category'] = df['textblob_polarity'].apply(classify_textblob_sentiment)

# Display example
df[['text', 'textblob_polarity', 'textblob_subjectivity', 'textblob_sentiment_category']].head()


,text,textblob_polarity,textblob_subjectivity,textblob_sentiment_category
0,Ignoring the specified pickup time. Providing ...,0.000000,0.000000,neutral
1,"Katrina, the tour guide was fabulous! She mad...",0.717000,0.728000,positive
2,A great app! My excursions have been awesome! ...,0.847222,0.861111,positive
3,I booked the excursion and paid the money up f...,0.125000,0.175000,positive
4,Great food tour of Prague,0.800000,0.750000,positive


### Why Are Sentiment Thresholds Different for VADER and TextBlob?

VADER’s compound score ranges from -1 to +1 and uses a neutral zone between -0.05 and 0.05 to filter out weak sentiment signals.  
- Positive: ≥ 0.05  
- Negative: ≤ -0.05  
- Neutral: between -0.05 and 0.05

TextBlob’s polarity also ranges from -1 to +1 but classifies strictly by sign without a neutral buffer:  
- Positive: > 0  
- Negative: < 0  
- Neutral: = 0

VADER’s threshold helps avoid misclassifying weak sentiment, while TextBlob uses a simpler binary cutoff.  
For consistency, you can apply VADER-like thresholds to TextBlob too.

## 4. Create Ground Truth Labels from Rating


In [12]:
def classify_rating_sentiment(rating):
    if pd.isna(rating):
        return None
    if rating > 4:
        return 'positive'
    elif rating <= 2:
        return 'negative'
    else:
        return 'neutral'

df['actual_sentiment_category'] = df['rating'].apply(classify_rating_sentiment)

# Preview
df[['text', 'rating', 'actual_sentiment_category', 'vader_sentiment_category', 'textblob_sentiment_category']].head()


,text,rating,actual_sentiment_category,vader_sentiment_category,textblob_sentiment_category
0,Ignoring the specified pickup time. Providing ...,1,negative,negative,neutral
1,"Katrina, the tour guide was fabulous! She mad...",5,positive,positive,positive
2,A great app! My excursions have been awesome! ...,5,positive,positive,positive
3,I booked the excursion and paid the money up f...,1,negative,negative,positive
4,Great food tour of Prague,5,positive,positive,positive


## 5. Classification Metrics: Accuracy, Confusion Matrix, F1-Score


In [13]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Filter out rows with missing sentiment labels
df_cleaned = df.dropna(subset=['actual_sentiment_category', 'vader_sentiment_category', 'textblob_sentiment_category'])

y_true = df_cleaned['actual_sentiment_category']
y_pred_vader = df_cleaned['vader_sentiment_category']
y_pred_textblob = df_cleaned['textblob_sentiment_category']


# Confusion Matrices
print("Confusion Matrix for TextBlob:")
print(confusion_matrix(y_true, y_pred_textblob, labels=['negative', 'neutral', 'positive']))

print("\nConfusion Matrix for VADER:")
print(confusion_matrix(y_true, y_pred_vader, labels=['negative', 'neutral', 'positive']))



Confusion Matrix for TextBlob:
[[ 2992   739  2476]
 [  273   209  1670]
 [  255   790 11694]]

Confusion Matrix for VADER:
[[ 4447   416  1344]
 [  387   250  1515]
 [  390   912 11437]]



### 🔍 Interpretation:

- **Higher accuracy across all classes** compared to TextBlob.
- Still some overprediction of `positive`, but less severe.
- **Better recall for `negative`**:
  - ~4,447 correctly identified negative reviews vs. only ~2,994 with TextBlob.
- **Neutral class still underperforming**:
  - Only 250 correct out of ~2,150, indicating both models struggle with detecting neutral tone.

---

### ✅ Conclusion

- **VADER outperforms TextBlob** in identifying `negative` and `neutral` reviews.
- **TextBlob is overly optimistic**, classifying a majority of reviews as positive.
- For multi-class sentiment classification, **VADER provides better class balance and predictive reliability**.


In [14]:
# VADER Accuracy & Report
accuracy_vader = accuracy_score(y_true, y_pred_vader)
print(f"Accuracy for VADER: {accuracy_vader:.2f}")
print("\nClassification Report for VADER:")
print(classification_report(y_true, y_pred_vader, zero_division=0))
print("------------------------------------------------------")

# TextBlob Accuracy & Report
accuracy_textblob = accuracy_score(y_true, y_pred_textblob)
print(f"\nAccuracy for TextBlob: {accuracy_textblob:.2f}")
print("\nClassification Report for TextBlob:")
print(classification_report(y_true, y_pred_textblob, zero_division=0))


Accuracy for VADER: 0.76

Classification Report for VADER:
              precision    recall  f1-score   support

    negative       0.85      0.72      0.78      6207
     neutral       0.16      0.12      0.13      2152
    positive       0.80      0.90      0.85     12739

    accuracy                           0.76     21098
   macro avg       0.60      0.58      0.59     21098
weighted avg       0.75      0.76      0.75     21098

------------------------------------------------------

Accuracy for TextBlob: 0.71

Classification Report for TextBlob:
              precision    recall  f1-score   support

    negative       0.85      0.48      0.62      6207
     neutral       0.12      0.10      0.11      2152
    positive       0.74      0.92      0.82     12739

    accuracy                           0.71     21098
   macro avg       0.57      0.50      0.51     21098
weighted avg       0.71      0.71      0.69     21098



### 🧠 Interpretation

- **Positive class**:
  - Both models perform well, especially in recall.
  - VADER is slightly more balanced between precision and recall.

- **Negative class**:
  - VADER: High precision *and* solid recall (72%) → strong performance.
  - TextBlob: High precision but low recall (48%) → many false negatives.

- **Neutral class**:
  - **Both models struggle.** F1-scores are below 0.15.
  - This indicates the models can't reliably detect neutral sentiment — possibly due to ambiguous or short reviews.

---

### ✅ Conclusion

- **VADER clearly outperforms TextBlob overall**, especially in detecting negative sentiment.
- **TextBlob is biased toward predicting positive**, which increases recall for positive but hurts balance.
- The **neutral class remains the weakest point** for both approaches — further tuning or model enhancement may be needed.

## 6. Regression Metrics: MAE, RMSE, R², Pearson


In [16]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

def map_rating_to_sentiment_score(rating):
    if pd.isna(rating):
        return None
    if rating == 1: return -1.0
    if rating == 2: return -0.5
    if rating == 3: return 0.0
    if rating == 4: return 0.5
    if rating == 5: return 1.0

df['actual_sentiment_score'] = df['rating'].apply(map_rating_to_sentiment_score)

df_cleaned = df.dropna(subset=['actual_sentiment_score', 'vader_compound', 'textblob_polarity'])

y_true_scores = df_cleaned['actual_sentiment_score']


# VADER Regression
y_pred_vader_scores = df_cleaned['vader_compound']

print("--- VADER Regression ---")
print(f"MAE: {mean_absolute_error(y_true_scores, y_pred_vader_scores):.4f}")
print(f"MSE: {mean_squared_error(y_true_scores, y_pred_vader_scores):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_true_scores, y_pred_vader_scores)):.4f}")
print(f"R²: {r2_score(y_true_scores, y_pred_vader_scores):.4f}")
print(f"Pearson: {pearsonr(y_true_scores, y_pred_vader_scores)[0]:.4f}")

print("------------------------------------------------------")

# TextBlob Regression
y_pred_textblob_scores = df_cleaned['textblob_polarity']

print("--- TextBlob Regression ---")
print(f"MAE: {mean_absolute_error(y_true_scores, y_pred_textblob_scores):.4f}")
print(f"MSE: {mean_squared_error(y_true_scores, y_pred_textblob_scores):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_true_scores, y_pred_textblob_scores)):.4f}")
print(f"R²: {r2_score(y_true_scores, y_pred_textblob_scores):.4f}")
print(f"Pearson: {pearsonr(y_true_scores, y_pred_textblob_scores)[0]:.4f}")



--- VADER Regression ---
MAE: 0.4410
MSE: 0.3644
RMSE: 0.6036
R²: 0.5284
Pearson: 0.7295
------------------------------------------------------
--- TextBlob Regression ---
MAE: 0.6496
MSE: 0.5261
RMSE: 0.7253
R²: 0.3190
Pearson: 0.6323



### 🧠 Interpretation

- **Lower MAE and RMSE** for VADER indicate it has smaller average errors when predicting sentiment scores.
- **R² = 0.53** for VADER means it explains over **50% of the variance** in the actual ratings — compared to only 32% for TextBlob.
- **Pearson correlation** of **0.73** vs. **0.63** shows that VADER's predictions are more linearly aligned with the true sentiment scores.
- Overall, **VADER outperforms TextBlob** across all metrics, especially in precision and reliability.

---

### ✅ Conclusion

For this dataset, **VADER is the more accurate and consistent tool** for predicting sentiment scores based on customer reviews.  
TextBlob still captures some sentiment trends, but is less precise — likely due to its simpler lexicon-based approach.


In [17]:
df

,name,country,rating,title,text,date_of_experience,has_reply,company,vader_sentiment,vader_compound,vader_neg,vader_neu,vader_pos,vader_sentiment_category,textblob_sentiment,textblob_polarity,textblob_subjectivity,textblob_sentiment_category,actual_sentiment_category,actual_sentiment_score
0,Przemysław Rosuł,PL,1,Ignoring the specified pickup time,Ignoring the specified pickup time. Providing ...,2025-05-13,0,www.viator.com,"{'neg': 0.317, 'neu': 0.625, 'pos': 0.058, 'co...",-0.7013,0.317,0.625,0.058,negative,"{'polarity': 0.0, 'subjectivity': 0.0}",0.000000,0.000000,neutral,negative,-1.0
1,kathy mrozek,US,5,Athens Evening food tour with Katrina,"Katrina, the tour guide was fabulous! She mad...",2025-05-15,0,www.viator.com,"{'neg': 0.048, 'neu': 0.715, 'pos': 0.238, 'co...",0.8682,0.048,0.715,0.238,positive,"{'polarity': 0.717, 'subjectivity': 0.728}",0.717000,0.728000,positive,positive,1.0
2,MARY MURRAY,US,5,All you need for travel with confidence,A great app! My excursions have been awesome! ...,2025-05-12,0,www.viator.com,"{'neg': 0.0, 'neu': 0.347, 'pos': 0.653, 'comp...",0.9440,0.000,0.347,0.653,positive,"{'polarity': 0.8472222222222223, 'subjectivity...",0.847222,0.861111,positive,positive,1.0
3,K Shay Smith,MX,1,I booked the excursion and paid the…,I booked the excursion and paid the money up f...,2025-05-12,0,www.viator.com,"{'neg': 0.137, 'neu': 0.863, 'pos': 0.0, 'comp...",-0.5574,0.137,0.863,0.000,negative,"{'polarity': 0.125, 'subjectivity': 0.175}",0.125000,0.175000,positive,negative,-1.0
4,Jeff Paine,US,5,Great food tour of Prague. Great conversation ...,Great food tour of Prague,2025-05-07,0,www.viator.com,"{'neg': 0.0, 'neu': 0.494, 'pos': 0.506, 'comp...",0.6249,0.000,0.494,0.506,positive,"{'polarity': 0.8, 'subjectivity': 0.75}",0.800000,0.750000,positive,positive,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21093,Asma,US,1,I rented a car with Expedia and paid…,I rented a car with Expedia and paid the money...,2025-02-08,0,www.expedia.com,"{'neg': 0.098, 'neu': 0.817, 'pos': 0.085, 'co...",0.4710,0.098,0.817,0.085,positive,"{'polarity': 0.5, 'subjectivity': 0.35}",0.500000,0.350000,positive,negative,-1.0
21094,Algirdas Dembinskas,US,1,Platinum user review,I’ve been using Expedia frequently for over 10...,2025-02-16,0,www.expedia.com,"{'neg': 0.196, 'neu': 0.728, 'pos': 0.076, 'co...",-0.6784,0.196,0.728,0.076,negative,"{'polarity': 0.0, 'subjectivity': 0.3650000000...",0.000000,0.365000,neutral,negative,-1.0
21095,Erin McLaughlin,US,1,NEVER book a flight through Expedia,NEVER book a flight through Expedia. “Fully re...,2025-02-15,0,www.expedia.com,"{'neg': 0.093, 'neu': 0.897, 'pos': 0.01, 'com...",-0.8718,0.093,0.897,0.010,negative,"{'polarity': 0.011363636363636354, 'subjectivi...",0.011364,0.425000,positive,negative,-1.0
21096,Lane Rosen,US,1,Expedia a Very deceptive company,Very deceptive company. Charge double the reg...,2025-02-14,0,www.expedia.com,"{'neg': 0.116, 'neu': 0.822, 'pos': 0.062, 'co...",-0.1531,0.116,0.822,0.062,negative,"{'polarity': 0.06666666666666667, 'subjectivit...",0.066667,0.125641,positive,negative,-1.0


In [18]:
df.to_csv('sentiment_analysis.csv', index=False)